In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from My_plotter import Plotter, Style
from T_method import LayeredStructure
from Global_optimizer import my_json_load
from pathlib import Path

In [ ]:
def y_l(y, beta):
    if y == np.inf:
        return (1)/(1.j * np.tan(beta))
    return (y + 1.j * np.tan(beta))/(1 + 1.j * y * np.tan(beta))
def r(y):
    if y == np.inf:
        return -1
    return (1-y)/(1+y)
def wrap_to_pi(angles):
    """Приводит углы к диапазону [-pi, pi)"""
    return (angles + np.pi) % (2 * np.pi) - np.pi

In [ ]:
# path = Path(f"kz/de_18.0_seed_90.json")
# data = my_json_load(path)
# alpha = np.array(data["best_alpha"])
# beta = np.array(data["best_beta"])
alpha = [7.401241886243108, 2.8245456388047856, 1.0954999271732788] 
beta = [2.514960899622451, 2.5359952073603886, 2.3220887543885342]
print(alpha)
print(beta)

In [ ]:
st = Style()
fig, ax = plt.subplots()
pl = Plotter(ax, st)
df = np.linspace(-50, 50, 200)/100
structure = LayeredStructure(alpha, beta=beta)
directivity = 10*np.log10(structure.directivity(df))
directivity_n = 6+10*np.log10(structure.directivity_naive(df))
pl.plot(df, directivity, label=f"16dBi, seed=90")
pl.plot(df, directivity_n)
pl.finalize()
pl.set_ylim((-10, 25))
ax.axhline(18, color='gray', linestyle='--', alpha=0.5)
ax.axvline(0.12086620866208656, color='gray', linestyle='--', alpha=0.5)
ax.axvline(0.06356356356356357, color='gray', linestyle='--', alpha=0.5)
ax.axvline(-0.08358358358358359, color='gray', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
def phase(df, k, alpha, beta): #k - индекс резонатора, начиная с нуля
    alpha = np.array(alpha)/(1+df) #пока только для индуктивных
    beta = np.array(beta)*(1+df)
    n = len(alpha)
    yl = np.inf
    yr = 1
    for i in np.arange(k):
        yl = y_l(yl, beta[i])
        yl = yl - alpha[i]*1.j #потому что y = alpha * (-1.j)
    rl = r(yl)
    yr = yr - alpha[n-1]*1.j
    for i in np.arange(k, n-1)[::-1]:
        yr = y_l(yr, beta[i+1])
        yr = yr - alpha[i]*1.j #потому что y = alpha * (-1.j)
    rr = r(yr)
    #print(df, 2*beta[k], np.angle(rl), np.angle(rr))
    return (-2*beta[k] + np.angle(rl) + np.angle(rr))

In [ ]:
st = Style()
fig, ax = plt.subplots()
pl = Plotter(ax, st)
df_arr = np.linspace(-50, 50, 1000)/100
for i in range(4):
    phase0 = np.array([phase(df, i, alpha, beta) for df in df_arr])
    phase0 = np.unwrap(phase0)
    pl.plot(df_arr, phase0)
pl.finalize()
ax.axvline(0.12086620866208656, color='gray', linestyle='--', alpha=0.5)
ax.axvline(0.06356356356356357, color='gray', linestyle='--', alpha=0.5)
ax.axvline(-0.018518518518518476, color='gray', linestyle='--', alpha=0.5)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.axhline(2*np.pi, color='gray', linestyle='--', alpha=0.5)
ax.axhline(-2*np.pi, color='gray', linestyle='--', alpha=0.5)
ax.axhline(-4*np.pi, color='gray', linestyle='--', alpha=0.5)
print(df_arr[np.argmin(np.abs(phase0))])

In [ ]:
alpha = [7.1355583870651245, 1.6746864995843964]
beta = [2.6216056734328594, 2.5039942175064516]

In [ ]:
def resonator_freqs(df, alpha, beta):
    freqs = []
    for k in range(len(alpha)):
        phase0 = np.array([phase(df, k, alpha, beta) for df in df_arr])
        phase0 = np.unwrap(phase0)
        freqs.append(df_arr[np.argmin(np.abs(phase0))])
    return freqs
print(resonator_freqs(df, alpha, beta))

In [ ]:
st = Style()
fig, ax = plt.subplots()
pl = Plotter(ax, st)
df = np.linspace(-50, 50, 400)/100
structure = LayeredStructure(alpha, beta=beta)
directivity = 10*np.log10(structure.directivity(df))
directivity_n = 6+10*np.log10(structure.directivity_naive(df))
pl.plot(df, directivity, label=f"16dBi, seed=90")
pl.plot(df, directivity_n)
pl.finalize()
pl.set_ylim((-10, 25))
ax.axhline(18, color='gray', linestyle='--', alpha=0.5)
for freq in resonator_freqs(df, alpha, beta):
    ax.axvline(freq, color='gray', linestyle='--', alpha=0.5)
plt.show()